In [3]:
import pandas as pd

df = pd.read_csv(r"C:\Users\shett\Downloads\fortune cleaned.csv")

# Combine important fields into text
df["text"] = df.apply(lambda row: 
    f"{row['Company']} is in {row['Sector']} sector, led by {row['CEO']} located in {row['HeadquartersCity']}, {row['HeadquartersState']}", axis=1)

texts = df["text"].tolist()
print(texts[:2])

['Walmart is in Retailing sector, led by C. Douglas Mcmillon located in Bentonville, Arkansas', 'Amazon is in Retailing sector, led by Andrew R. Jassy located in Seattle, Washington']


In [6]:
from langchain_openai import OpenAIEmbeddings

In [8]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(texts)

print(len(embeddings), len(embeddings[0]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\shett\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shett\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

1000 384


In [9]:
import faiss
import numpy as np

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors:", index.ntotal)

Total vectors: 1000


In [15]:
def semantic_search(query, k=3):
    query_embedding = model.encode([query])

    D, I = index.search(query_embedding, k)

    results = [texts[i] for i in I[0]]
    return results

print(semantic_search("technology companies"))

['Cognizant Technology Solutions is in Technology sector, led by Ravi Kumar S located in Teaneck, New Jersey', 'Ss&C Technologies Holdings is in Technology sector, led by William C. Stone located in Windsor, Connecticut', 'Enterprise Products Partners is in Energy sector, led by A. James Teague/W. Randall Fowler located in Houston, Texas']


In [18]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pcsk_4d4HZw_6oSoCyvY9d26nptmnujm5wPVfU7BckK84FHprjVmXa6KoodFj8FySJiNRF14gf4")

index_name = "company-search"

# Create index (only once)
if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print("✅ Connected to Pinecone")

✅ Connected to Pinecone


In [20]:
vectors = []

for i, emb in enumerate(embeddings):
    vectors.append((
        str(i),              # unique ID
        emb.tolist(),        # embedding vector
        {"text": texts[i]}   # metadata
    ))

print(vectors[:2])   # check sample

[('0', [0.051493529230356216, -0.0005534724332392216, -0.08437597006559372, -0.015103927813470364, 0.08571650087833405, 0.05001209303736687, -0.05948140472173691, -0.034171659499406815, -0.04125760868191719, -0.002631962765008211, 0.05414515361189842, 0.04970654472708702, -0.012176756747066975, -0.05693987384438515, 0.05881040170788765, -0.033691536635160446, 0.03709305822849274, -0.013324256986379623, 0.0692666545510292, -0.08581044524908066, 0.0014844738179817796, -0.0390661247074604, -0.01197128091007471, 0.001783978077583015, -0.10984215140342712, -0.024290619418025017, 0.050631798803806305, -0.041684817522764206, -0.02152355946600437, -0.018913734704256058, -0.0034538416657596827, -0.025059064850211143, 0.08903776854276657, 0.06317361444234848, 0.05250569432973862, 0.061482589691877365, 0.05608205124735832, -0.006365717854350805, 0.08539106696844101, -0.025800172239542007, 0.09225599467754364, 0.01215516496449709, -0.02485828846693039, -0.10706942528486252, -0.03904455155134201, -

In [22]:
index.upsert(vectors)

print("✅ Data stored in Pinecone")

✅ Data stored in Pinecone


In [29]:
def pinecone_search(query, k=3):
    query_vector = model.encode([query]).tolist()

    results = index.query(
        vector=query_vector[0],
        top_k=k,
        include_metadata=True
    )

    return [match['metadata']['text'] for match in results['matches']]

In [31]:
from transformers import pipeline

# Load free model
generator = pipeline("text-generation", model="gpt2")

def rag_llm(query):
    # Step 1: Retrieve
    docs = pinecone_search(query)

    # Step 2: Context
    context = "\n".join(docs)

    # Step 3: Prompt
    prompt = f"""
    Context:
    {context}

    Question: {query}
    Answer:
    """

    # Step 4: Generate
    result = generator(prompt, max_length=200, num_return_sequences=1)

    return result[0]['generated_text']


print(rag_llm("Which companies are in technology sector?"))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\shett\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shett\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    Context:
    Cognizant Technology Solutions is in Technology sector, led by Ravi Kumar S located in Teaneck, New Jersey
Ibm is in Technology sector, led by Arvind Krishna located in Armonk, New York
Dell Technologies is in Technology sector, led by Michael S. Dell located in Round Rock, Texas

    Question: Which companies are in technology sector?
    Answer:
                               
Jobs In Technology:                               
New York City Technology Management:                                       
Venture Capital Partners is in Technology sector, led by Daniel D. Venture Capital Partners located in Port Washington, Rhode Island
New York City Technology Management:                                                 
Nasdaq Venture Capital is in Technology sector, led by Brian R. Nasdaq Venture Capital is in Technology sector, led by Andrew R. Neskope located in New York City

                         


In [5]:
import langchain
print(langchain.__version__)

1.2.12
